In [1]:
import os

import boto3
import pandas as pd
from sqlalchemy import create_engine, text

POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ROOT_USER = os.getenv("MINIO_ROOT_USER")
MINIO_ROOT_PASSWORD = os.getenv("MINIO_ROOT_PASSWORD")
MINIO_BUCKET = os.getenv("MINIO_BUCKET")

print("POSTGRES_HOST:", POSTGRES_HOST)
print("POSTGRES_PORT:", POSTGRES_PORT)
print("POSTGRES_DB:", POSTGRES_DB)
print("MINIO_ENDPOINT:", MINIO_ENDPOINT)
print("MINIO_BUCKET:", MINIO_BUCKET)

POSTGRES_HOST: postgres
POSTGRES_PORT: 5432
POSTGRES_DB: oil_industry
MINIO_ENDPOINT: http://minio:9000
MINIO_BUCKET: datalake


In [2]:
db_url = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

engine = create_engine(db_url)

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))
    tables = [row[0] for row in result]

tables

['deliveries',
 'drivers',
 'production',
 'pump_failures',
 'pump_sensors',
 'pumps',
 'vehicles',
 'well_targets',
 'well_telemetry',
 'wells']

In [3]:
df = pd.read_sql("""
    SELECT *
    FROM production
    LIMIT 5;
""", engine)

df

,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9


In [4]:
s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ROOT_USER,
    aws_secret_access_key=MINIO_ROOT_PASSWORD,
)

response = s3_client.list_buckets()

[bucket["Name"] for bucket in response["Buckets"]]

['datalake']

In [5]:
s3_client.put_object(
    Bucket=MINIO_BUCKET,
    Key="test/connection_check.txt",
    Body=b"connection check",
)

objects = s3_client.list_objects_v2(
    Bucket=MINIO_BUCKET,
    Prefix="test/"
)

[obj["Key"] for obj in objects.get("Contents", [])]

['test/connection_check.txt']